###Se realizara el muestreo, vectorización y tokenización del dataset

In [22]:
import pandas as pd
import os
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist
import numpy as np
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
ruta_train = '/content/drive/My Drive/twitter_analysis_dataset/training_twitter.csv'
ruta_test = '/content/drive/My Drive/twitter_analysis_dataset/testdata_manual.csv'

# Cargar
df_train = pd.read_csv(ruta_train, encoding='latin-1', header=None)
df_train.columns = ['target', 'id', 'date', 'query', 'user', 'text']
df_test = pd.read_csv(ruta_test, header=None)
df_test.columns = ['target', 'id', 'date', 'query', 'user', 'text']

In [24]:
# 2. Limpieza
def limpiar_tweet(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

In [25]:
# 1. Defines la lista de columnas que vas a borrar (sirve para ambos)
columnas_a_eliminar = ['id', 'user', 'query', 'date']

# 2. Le aplicas el "drop" al DataFrame de entrenamiento
df_train.drop(columns=columnas_a_eliminar, inplace=True)

# 3. Le aplicas el "drop" al DataFrame de prueba
df_test.drop(columns=columnas_a_eliminar, inplace=True)

In [26]:
# 1. Limpieza de todo el dataset
df_train['text_limpio'] = df_train['text'].apply(limpiar_tweet)

# 2. Muestreo (usando random_state para reproducibilidad)
df_muestra = df_train.sample(n=5000, random_state=42)

# 3. Vectorización (solo sobre la muestra)
vectorizer = TfidfVectorizer(max_features=200) # Un vocabulario más pequeño ayuda al clustering
X_muestra = vectorizer.fit_transform(df_muestra['text_limpio'])

In [29]:
# 1. Limpiar el dataset de prueba primero
df_test['text_limpio'] = df_test['text'].apply(limpiar_tweet)

# 2. Muestreo (ajustado al tamaño real de df_test)
tamano_muestra = min(len(df_test), 5000)
df_muestra_test = df_test.sample(n=tamano_muestra, random_state=42)

# 3. Vectorización (SOLO transformamos usando el vocabulario aprendido en train)
X_test_tfidf = vectorizer.transform(df_muestra_test['text_limpio'])

In [30]:
print("Columnas en TRAIN:", df_train.columns)
print("Columnas en TEST:", df_test.columns)

Columnas en TRAIN: Index(['target', 'text', 'text_limpio'], dtype='object')
Columnas en TEST: Index(['target', 'text', 'text_limpio'], dtype='object')


In [31]:
# 1. Defines la lista de columnas que vas a borrar (sirve para ambos)
columnas_a_eliminar = ['text']

# 2. Le aplicas el "drop" al DataFrame de entrenamiento
df_train.drop(columns=columnas_a_eliminar, inplace=True)

# 3. Le aplicas el "drop" al DataFrame de prueba
df_test.drop(columns=columnas_a_eliminar, inplace=True)

# 4. (Opcional) Verificas que ambos quedaron iguales
print("Columnas en TRAIN:", df_train.columns)
print("Columnas en TEST:", df_test.columns)

Columnas en TRAIN: Index(['target', 'text_limpio'], dtype='object')
Columnas en TEST: Index(['target', 'text_limpio'], dtype='object')


EN ESTE CUADERNO SE ELIMINARON LAS COLUMNAS QUE SE IDENTIFICARON EN LA OBSERVACIÓN COMO DATOS CONSTANTES QUE SOLO AÑADIRIAN RUIDO AL APRENDIZAJE.
(USER, ID, QUERY)
TAMBIEN SE GENERO UNA COLUMNA CON LA LIMPIEZA DE LOS TEXTOS ELIMINANDO SIMBOLOS ESPECIALES Y SE ELIMINO EL TEXTO ORIGINAL.

In [34]:
ruta_guardado = '/content/drive/My Drive/Colab Notebooks/TP3_DIPLO/Processed/'
df_train.to_csv(ruta_guardado + 'train_limpio.csv', index=False)
df_test.to_csv(ruta_guardado + 'test_limpio.csv', index=False)
print("¡Archivos guardados con éxito en Drive!")

¡Archivos guardados con éxito en Drive!
